# DAVID-Net Training â€” Kaggle

In [ ]:
# Cell 1: GPU check + install deps
import os
# --- keep this notebook's output small -------------------------------------------------
# An Interactive Kaggle session is tethered to the browser websocket, and Kaggle reaps the
# container when that connection stops heartbeating. The surest way to break it is an
# output flood: huggingface_hub upload bars and transformers' "Loading weights" emit
# thousands of carriage-return redraws per run. Silence them BEFORE anything imports them
# (subprocesses inherit os.environ, so this covers Cells 9 and 11 too).
os.environ['PYTHONUNBUFFERED'] = '1'   # stream subprocess logs line by line (no 8 KB pipe buffering)
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_HUB_VERBOSITY'] = 'error'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['TQDM_DISABLE'] = '1'
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU.")
!pip install -q transformers accelerate scikit-learn jiwer datasets
import shutil; assert shutil.which('ffmpeg'), 'ffmpeg missing - decoding needs it'
!df -h /dev/shm /kaggle/working | tail -n +1

In [ ]:
# Cell 2: Clone repo
import os, sys
REPO = "/kaggle/working/david-net-av"
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/MIHMahmudEli/david-net-av.git {REPO}
sys.path.insert(0, REPO)
print(f"Repo: {REPO}")

In [ ]:
# Cell 3: Load HF token + start the SESSION watchdog
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("HF token loaded.")

# Session-level telemetry -> HF runs/session_<id>/logs/watchdog_session.jsonl, one line per
# minute (RAM, VRAM, disk, /dev/shm, GPU util, ffmpeg count, current cell). Lives in the
# notebook kernel: if the TRAINING process dies, this keeps reporting; if THIS stops, the
# session itself was killed and its last line is the time of death.
# Name THIS account so `publish_campaign.py status` shows who holds each lease. Kaggle
# does not expose the owner in a batch session, so set it per account (any short label).
WORKER_NAME = ""          # e.g. "acct-01"; blank falls back to the container hostname
if WORKER_NAME:
    os.environ["DAVIDNET_WORKER"] = WORKER_NAME

import time as _time
SESSION_ID = _time.strftime("%Y%m%d_%H%M%S")
CURRENT_CELL = {"name": "cell3"}
def _on_pre_run(info):
    src = (info.raw_cell or "").strip().splitlines()
    CURRENT_CELL["name"] = src[0][:60] if src else "?"
get_ipython().events.register("pre_run_cell", _on_pre_run)
from src.utils.watchdog import start_session_watchdog
SESSION_WD = start_session_watchdog(SESSION_ID, local_dir="/kaggle/working",
                                    current_cell=lambda: CURRENT_CELL["name"])
RUN_TYPE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "?")
print(f"session watchdog started: runs/session_{SESSION_ID}/logs/watchdog_session.jsonl  "
      f"(run type: {RUN_TYPE})")
if RUN_TYPE != "Batch":
    print("\n" + "!" * 78)
    print("!! INTERACTIVE SESSION - this container dies with your browser tab.")
    print("!! A reap looks exactly like a silent stop: healthy RAM/VRAM/disk, no traceback,")
    print("!! no signal, both watchdogs ending in the same minute. Stage 1 is ~30 GPU-hours")
    print("!! and cannot finish here regardless.")
    print("!! For training, use:  Save Version -> Save & Run All (Commit)")
    print("!" * 78 + "\n")


In [ ]:
# Cell 4: Discover datasets; download only the ones this notebook declared it needs
import shutil
import subprocess
from pathlib import Path

# A notebook that needs a subset sets DOWNLOAD_ALLOWLIST before this cell. Default is
# everything, which is what the training notebook wants. The cache builder sets
# {"fakeavceleb"}: told to attach that alone, this cell used to start downloading the
# other six -- one of them 96.5 GB -- into a 20 GB disk, and the session died on ENOSPC
# before a single clip was decoded.
DOWNLOAD_ALLOWLIST = globals().get("DOWNLOAD_ALLOWLIST", None)
MIN_FREE_GB_TO_DOWNLOAD = 12.0

KAGGLE_INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
DOWNLOAD_DIR = WORKING / "kaggle_datasets"
DOWNLOAD_DIR.mkdir(exist_ok=True)

VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv"}
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".ogg"}

def has_media_files(d):
    for f in d.rglob("*"):
        if f.suffix.lower() in VIDEO_EXTS | AUDIO_EXTS:
            return True
    return False

DATASETS = {
    "fakeavceleb": {
        "slug": "aicontentdetections/fakeavceleb-v1-2",
        "mounts": ["aicontentdetections/fakeavceleb-v1-2", "aicontentdetections/FakeAVCeleb_v1.2"],
    },
    "dfdc-10": {
        "slug": "pranay22077/dfdc-10",
        "mounts": ["pranay22077/dfdc-10"],
    },
    "deepfaketimit": {
        "slug": "fahimaislam1812/deepfaketimit",
        "mounts": ["fahimaislam1812/deepfaketimit", "fahimaislam1812/DeepfakeTIMIT"],
    },
    "celeb-df-v2": {
        "slug": "reubensuju/celeb-df-v2",
        "mounts": ["reubensuju/celeb-df-v2"],
    },
    "asvpoof-2019": {
        "slug": "anishsarkar22/asvpoof-2019-dataset-la",
        "mounts": ["anishsarkar22/asvpoof-2019-dataset-la"],
    },
    "in-the-wild": {
        "slug": "abdallamohamed312/in-the-wild-audio-deepfake",
        "mounts": ["abdallamohamed312/in-the-wild-audio-deepfake"],
    },
    "wavefake": {
        "slug": "walimuhammadahmad/fakeaudio",
        "mounts": ["walimuhammadahmad/fakeaudio", "andreadiubaldo/wavefake-test"],
    },
}

if DOWNLOAD_ALLOWLIST is None:
    DOWNLOAD_ALLOWLIST = set(DATASETS)


def find_mounted(slug_paths):
    for p in slug_paths:
        for base in [KAGGLE_INPUT, KAGGLE_INPUT / "datasets"]:
            candidate = base / p
            if candidate.exists():
                return candidate
    return None

def download_dataset(slug, friendly):
    dst = DOWNLOAD_DIR / friendly
    if dst.exists() and has_media_files(dst):
        print(f"  {friendly}: already downloaded")
        return dst
    free_gb = shutil.disk_usage(str(WORKING)).free / 1e9
    if free_gb < MIN_FREE_GB_TO_DOWNLOAD:
        print(f"  {friendly}: SKIPPED -- only {free_gb:.1f} GB free, need "
              f"{MIN_FREE_GB_TO_DOWNLOAD:.0f} GB of headroom")
        return None
    dst.mkdir(parents=True, exist_ok=True)
    print(f"  {friendly}: downloading from {slug} ({free_gb:.1f} GB free)...", end=" ")
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "download", "-d", slug, "-p", str(dst), "--unzip"],
            capture_output=True, text=True, timeout=3600
        )
        if result.returncode == 0:
            print("OK")
            return dst
        else:
            print(f"FAIL: {result.stderr[:200]}")
            return None
    except Exception as e:
        print(f"ERROR: {e}")
        return None

datasets = {}
for friendly, info in DATASETS.items():
    mounted = find_mounted(info["mounts"])
    if mounted:
        datasets[friendly] = mounted
        print(f"  {friendly} -> {mounted} (mounted)")
        continue
    downloaded = DOWNLOAD_DIR / friendly
    if downloaded.exists() and has_media_files(downloaded):
        datasets[friendly] = downloaded
        print(f"  {friendly} -> {downloaded} (cached)")
        continue
    if friendly not in DOWNLOAD_ALLOWLIST:
        print(f"  {friendly}: not mounted, not needed by this notebook -> skipped")
        continue
    result = download_dataset(info["slug"], friendly)
    if result:
        datasets[friendly] = result

print(f"\nFound {len(datasets)}/{len(DATASETS)} datasets "
      f"({len(DOWNLOAD_ALLOWLIST)} were eligible to download).")
_missing = set(DATASETS) - set(datasets)
if _missing:
    print(f"Not present: {_missing}")
print(f"disk: {shutil.disk_usage(str(WORKING)).free / 1e9:.1f} GB free in {WORKING}")

In [ ]:
# Cell 5: Extract compressed datasets (multi-part zips, tar, etc.)
import zipfile, tarfile, shutil

WORKING = Path("/kaggle/working")
DATA_DIR = WORKING / "data"
DATA_DIR.mkdir(exist_ok=True)

def find_first_zip_part(src):
    """Find the .001 part of a multi-part zip, anywhere in tree."""
    best = None
    best_num = 999999
    count = 0
    for f in src.rglob("*"):
        name = f.name
        # Match patterns like: foo.zip.001, foo.zip.016
        if ".zip." in name:
            parts = name.split(".zip.")
            if len(parts) == 2 and parts[1].isdigit():
                count += 1
                num = int(parts[1])
                if num < best_num:
                    best_num = num
                    best = f
    if best:
        print(f"    Found {count} zip parts, first = {best.name} (part {best_num})")
    return best, count

def extract_multipart_zip(name, first_part, dst, search_root=None):
    print(f"  {name}: extracting multi-part zip from {first_part.name}...")
    if search_root is None:
        search_root = first_part.parent
    stem = first_part.name.split(".zip.")[0]
    # Find ALL parts across all subdirs
    all_parts = sorted(search_root.rglob(f"{stem}.zip.*"),
                       key=lambda x: int(x.name.split(".zip.")[1]))
    print(f"  {name}: found {len(all_parts)} parts across subdirs")

    # Method 1: Try 7z â€” copy all parts to temp dir first (7z needs them co-located)
    try:
        import shutil, tempfile
        with tempfile.TemporaryDirectory() as tmpdir:
            tmp = Path(tmpdir)
            for p in all_parts:
                shutil.copy2(str(p), str(tmp / p.name))
            first_tmp = tmp / first_part.name
            result = subprocess.run(
                ["7z", "x", str(first_tmp), f"-o{dst}", "-y"],
                capture_output=True, text=True, timeout=600
            )
            if result.returncode == 0:
                print(f"  {name}: OK (via 7z)")
                return True
            print(f"  {name}: 7z failed: {result.stderr[:200]}")
    except FileNotFoundError:
        print(f"  {name}: 7z not found, trying concat method...")
    except subprocess.TimeoutExpired:
        print(f"  {name}: 7z timed out")

    # Method 2: Concatenate all parts into one zip, then extract
    # Search from dataset root (not just .001 parent) since parts may be in sibling dirs
    # Method 2: Concatenate all parts into one zip, then extract
    try:
        print(f"  {name}: concatenating {len(all_parts)} parts...")
        merged = dst / f"{stem}_merged.zip"
        with open(merged, "wb") as out:
            for part in all_parts:
                with open(part, "rb") as inp:
                    while True:
                        chunk = inp.read(8 * 1024 * 1024)
                        if not chunk:
                            break
                        out.write(chunk)
        print(f"  {name}: merged to {merged.stat().st_size // 1024 // 1024}MB, extracting...")
        with zipfile.ZipFile(str(merged)) as zf:
            zf.extractall(dst)
        merged.unlink()  # remove merged zip to save space
        print(f"  {name}: OK (via concat)")
        return True
    except Exception as e:
        print(f"  {name}: FAILED: {e}")
        return False

def extract_archives(name, src, dst):
    extracted = False
    # Multi-part zip first â€” pass src as search_root so we find parts across all subdirs
    first_part, count = find_first_zip_part(src)
    if first_part:
        extracted = extract_multipart_zip(name, first_part, dst, search_root=src)
    # Regular zips
    for arch in src.rglob("*.zip"):
        if ".zip." in arch.name:
            continue
        print(f"  {name}: extracting {arch.name}...", end=" ")
        try:
            with zipfile.ZipFile(arch) as zf:
                zf.extractall(dst)
            print("OK")
            extracted = True
        except Exception as e:
            print(f"FAIL: {e}")
    # Tar.gz
    for arch in src.rglob("*.tar.gz"):
        print(f"  {name}: extracting {arch.name}...", end=" ")
        try:
            with tarfile.open(arch, "r:gz") as tf:
                tf.extractall(dst)
            print("OK")
            extracted = True
        except Exception as e:
            print(f"FAIL: {e}")
    return extracted

for name, path in datasets.items():
    dst = DATA_DIR / name
    dst.mkdir(exist_ok=True)
    marker = dst / ".extracted"
    if marker.exists():
        print(f"  {name}: already extracted")
        continue
    if has_media_files(path):
        print(f"  {name}: loose media files found")
        marker.touch()
        continue
    # Check disk space â€” skip if dataset too large for Kaggle (~20GB working)
    import shutil as _shutil
    free_gb = _shutil.disk_usage(str(DATA_DIR)).free / (1024**3)
    zip_count = sum(1 for _ in path.rglob("*.zip.*") if '.zip.' in _.name and _.name.split('.zip.')[1].isdigit())
    est_gb = zip_count * 1.0  # each part is ~1GB
    if est_gb > free_gb * 0.85:
        print(f"  {name}: SKIPPED â€” {est_gb:.0f}GB needed but only {free_gb:.1f}GB free")
        continue
    print(f"  {name}: no loose media, searching archives...")
    try:
        extract_archives(name, path, dst)
    except OSError as e:
        print(f"  {name}: FAILED (disk error): {e}")
        # Clean up partial extraction to free space
        import shutil as _shutil2
        if dst.exists():
            _shutil2.rmtree(dst, ignore_errors=True)
        continue
    if has_media_files(dst):
        print(f"  {name}: extracted media OK")
    else:
        print(f"  {name}: WARNING - no media files after extraction")
    marker.touch()

print("\nExtraction done.")

In [ ]:
# Cell 6: Build ALL manifests
MANIFEST_DIR = WORKING / "manifests"
MANIFEST_DIR.mkdir(exist_ok=True)
SPLIT_DIR = WORKING / "splits"
SPLIT_DIR.mkdir(exist_ok=True)

# Campaign determinism: a mounted Kaggle dataset always resolves to its LATEST version, so
# rebuilding manifests on a different account can silently produce a DIFFERENT subject
# split -- incomparable seeds, and clips that are test here but train there. Once the
# campaign partition is published, never rebuild it; Cell 7 fetches and hash-verifies it.
from src.utils.splits_sync import published_index
SPLITS_PUBLISHED = published_index() is not None
print("campaign splits already published on HF -> skipping rebuild"
      if SPLITS_PUBLISHED else "no campaign splits yet -> building (Cell 7 publishes them)")

# === FakeAVCeleb ===
fakeav_root = datasets.get("fakeavceleb")
if fakeav_root:
    for candidate in [fakeav_root, fakeav_root / "FakeAVCeleb_v1.2"]:
        if any((candidate / q).exists() for q in ["RealVideo-RealAudio", "FakeVideo-FakeAudio"]):
            fakeav_root = candidate
            break
    # The manifest's rel_path is relative to THIS directory -> training must use the
    # same root. (Run 1 passed the mount root instead; every clip was "missing" and the
    # loader silently trained on random tensors. The loader now raises instead.)
    datasets["fakeavceleb"] = fakeav_root
    print(f"FakeAVCeleb: {fakeav_root}")
    if not SPLITS_PUBLISHED:
        !cd {REPO} && python scripts/build_manifest.py \
        --root {fakeav_root} \
        --out {MANIFEST_DIR}/fakeavceleb.jsonl \
        --splits-dir {SPLIT_DIR}/fakeavceleb --seed 42

# === All other datasets ===
CONVERTERS = [
    ("dfdc-10", "dfdc-10"),
    ("deepfaketimit", "deepfaketimit"),
    ("celeb-df-v2", "celeb-df-v2"),
    ("asvpoof-2019", "asvpoof-2019"),
    ("in-the-wild", "in-the-wild"),
    ("wavefake", "wavefake"),
]

for ds_name, dataset_key in CONVERTERS:
    root = datasets.get(dataset_key)
    extracted_root = DATA_DIR / ds_name
    if not root or not root.exists():
        if extracted_root.exists() and has_media_files(extracted_root):
            root = extracted_root
            print(f"  {ds_name}: using extracted path {root}")
    if root and root.exists():
        print(f"\n--- {ds_name} ---")
        if not SPLITS_PUBLISHED:
            !cd {REPO} && python scripts/build_manifests.py \
            --dataset {dataset_key} \
            --root {root} \
            --out {MANIFEST_DIR}/{ds_name}.jsonl \
            --splits-dir {SPLIT_DIR}/{ds_name}
    else:
        print(f"  {ds_name}: NOT FOUND")

print("\n" + "="*50)
print("ALL MANIFESTS:")
for f in sorted(MANIFEST_DIR.glob("*.jsonl")):
    !wc -l {f}

In [ ]:
# Cell 7: Training manifests = the ONE published, hash-verified campaign partition
import json, yaml
from collections import Counter
from src.utils.splits_sync import fetch_and_verify, published_index, publish

FAKEAV_ROOT = str(datasets["fakeavceleb"])   # the mount; see the cache cell below

# NOTE: an earlier version copied FakeAVCeleb into /kaggle/working first. Do not
# bring that back. Copying 6.6 GB charged ~20 GB to the container's 32 GB cgroup --
# the bytes read from the mount, the dirty pages written to /dev/loop2, and the loop
# device's own backing cache -- leaving training 4 GB of headroom and earning an
# immediate SIGKILL (exit 137). Clips are read straight from the mount now, and only
# once, by the cache builder.

# The first worker in a campaign freezes the partition; every later worker -- on any
# account -- downloads exactly those bytes and verifies SHA-256 before touching a GPU.
# SPLITS_SHA256.json is also what the paper cites so reviewers can reproduce the split.
if published_index() is None:
    print("publishing this session's splits as the campaign partition...")
    publish(str(SPLIT_DIR), extra_provenance={"fakeav_root": FAKEAV_ROOT})

VERIFIED_SPLITS = WORKING / "splits_verified"
fetch_and_verify(str(VERIFIED_SPLITS))              # raises SystemExit on any mismatch

FAKEAV_SPLITS = VERIFIED_SPLITS / "fakeavceleb"
TRAIN_MANIFEST = FAKEAV_SPLITS / "train.jsonl"
VAL_MANIFEST = FAKEAV_SPLITS / "val.jsonl"
TEST_MANIFEST = FAKEAV_SPLITS / "test.jsonl"
for m in (TRAIN_MANIFEST, VAL_MANIFEST, TEST_MANIFEST):
    assert m.exists(), f"missing split {m} - the published partition is incomplete"

def _summary(path):
    recs = [json.loads(l) for l in open(path) if l.strip()]
    return len(recs), dict(Counter(r["quadrant"] for r in recs)), dict(Counter(r["generator"] for r in recs))

for name, m in [("train", TRAIN_MANIFEST), ("val", VAL_MANIFEST), ("test", TEST_MANIFEST)]:
    n, quads, gens = _summary(m)
    print(f"{name:5s}: {n:6d} clips  quadrants={quads}")
    if name == "train":
        print(f"       generators={gens}")

# Hard preflight: the first train clip must exist under FAKEAV_ROOT and decode to real
# frames/audio (this is the check that would have caught run 1 at t=0).
first = json.loads(open(TRAIN_MANIFEST).readline())
probe = Path(FAKEAV_ROOT) / first["rel_path"]
assert probe.exists(), f"{probe} does not exist -> FAKEAV_ROOT is wrong"
from src.data.decode import decode_clip
d = decode_clip(str(probe), 16, 64000, 224, window="center")
print(f"preflight OK: {probe.name} video{tuple(d.video.shape)} std={d.video.std():.3f} "
      f"audio{tuple(d.audio.shape)} std={d.audio.std():.4f} has_audio={d.has_audio}")
assert d.has_audio and d.audio.std() > 1e-4 and d.video.std() > 1e-3, "decoded clip looks empty"

In [ ]:
# Cell 8: the packed clip cache -- the media path that survives Kaggle's memory cgroup
#
# Stage 1 used to run ffmpeg inside every DataLoader worker. Eight workers, each forking
# a subprocess and holding decoded float32 frames, pushed the container past its 32 GB
# cgroup limit about five minutes into every run; the kernel answered with SIGKILL and
# Kaggle reported "Canceled by backend, exit code 137" with no log and no checkpoint.
#
# So every clip is decoded ONCE into ~480 KB of tiled JPEG + int16 PCM, packed into a
# handful of large shards. Training then does one pread and one JPEG decode per sample:
# no subprocess, no decode timeout, and only the few hundred KB actually read ends up in
# page cache -- clean, file-backed and reclaimable.
#
# Build it once (see kaggle_kernel/build_cache.ipynb, a CPU session that spends no GPU
# quota), publish it as a Kaggle Dataset, then attach that dataset on all 10 accounts.
from pathlib import Path

CACHE_DATASET_SLUG = "davidnet-av-cache"     # rename if you publish under another slug

def _find_mounted_cache():
    """A cache dataset attached to this kernel, if any."""
    base = Path("/kaggle/input")
    if not base.exists():
        return None
    for idx in sorted(base.glob("*/index.json")) + sorted(base.glob("*/*/index.json")):
        try:
            meta = json.loads(idx.read_text())
        except Exception:
            continue
        if meta.get("format") == "DVC2" and meta.get("clips"):
            return idx.parent
    return None

CACHE_ROOT = None
_mounted = _find_mounted_cache()
if _mounted is not None:
    CACHE_ROOT = str(_mounted)
    print(f"clip cache (mounted, read-only): {CACHE_ROOT}")
else:
    _local = WORKING / "av_cache"
    if (_local / "index.json").exists():
        CACHE_ROOT = str(_local)
        print(f"clip cache (local): {CACHE_ROOT}")
    else:
        print(f"NO CLIP CACHE FOUND.\n"
              f"  Add the '{CACHE_DATASET_SLUG}' dataset to this notebook "
              f"(+ Add Input -> Datasets), or run kaggle_kernel/build_cache.ipynb once.\n"
              f"  Falling back to live ffmpeg decoding -- this is the configuration that "
              f"was killed at ~5 minutes. Expect it to fail.")

if CACHE_ROOT:
    from src.data.clipcache import ClipCache
    _c = ClipCache(CACHE_ROOT)
    _recs = [json.loads(l) for l in open(TRAIN_MANIFEST) if l.strip()]
    _cov = _c.coverage(_recs)
    print(f"  {len(_c)} clips cached; train-split coverage {_cov * 100:.2f}%"
          + (f"; {len(_c.failures)} undecodable" if _c.failures else ""))
    _v, _a, _hv, _ha = _c.read(_recs[0]["clip_id"], 16, 64000, "center")
    print(f"  preflight: video{tuple(_v.shape)} std={_v.std():.3f} "
          f"audio{tuple(_a.shape)} std={_a.std():.4f} has_audio={_ha}")
    assert _v.std() > 1e-3 and _a.std() > 1e-5, "cached clip decodes to something empty"
    if _cov < 0.98:
        print("  WARNING coverage below 98% -- finish the cache build before training")

# With the cache, a sample costs one pread plus one JPEG decode (~20 ms), not an ffmpeg
# subprocess. Two workers keep two T4s fed; more only buys back the memory pressure.
NUM_WORKERS = 2 if CACHE_ROOT else 4
print(f"num_workers = {NUM_WORKERS}")

In [ ]:
# Cell 8: QACP Stage 0 config (pristine RVRA clips of the TRAIN split only)
QACP_CONFIG = {
    "run_id": "qacp_stage0_v4",          # v4: batch_size 8. v3 ran at 4, where 1 batch in 5
                                        # left the sync axis with no negative and qacp_c sat on
                                        # ln(3) all run. Do NOT resume v3: its optimizer state and
                                        # LR schedule belong to the old batch size.
    "d_model": 768, "n_heads": 8, "n_fusion_layers": 4, "dropout": 0.1,
    "use_sync": True, "use_disentangle": True, "compose_quadrant": False,
    "video_backbone": "videomae", "audio_backbone": "wavlm",
    "video_model_name": "MCG-NJU/videomae-base", "audio_model_name": "microsoft/wavlm-base-plus",
    "freeze_blocks": 8, "freeze_feature_extractor": True,
    "init_from": None,
    "n_frames": 16, "audio_len": 64000, "shard_root": None, "feature_cache": None,
    "cache_root": CACHE_ROOT,       # packed clips; None = live ffmpeg decode
    "train_manifest": str(TRAIN_MANIFEST),          # filtered to RVRA inside the script
    "root_dir": FAKEAV_ROOT,
    "qacp_views_per_clip": 4,                        # ~350 reals x 4 pseudo-views per epoch
    # batch_size 8, NOT 4. SupCon negatives live inside the micro-batch, so accumulation
    # cannot help. At 4, one batch in five gave the sync axis no negative at all and
    # qacp_c sat on ln(3)=1.0986 for every step of every run; at 8 that rate is 0%
    # (scripts/probe_qacp_batch.py). Measured on a T4: 9.15 GB of 15.6; 16 OOMs.
    # grad_accum 2 -> 1 keeps 175 optimizer steps/epoch, so the LR schedule is unchanged.
    "batch_size": 8, "grad_accum_steps": 1, "num_workers": NUM_WORKERS,
    "epochs": 15,
    "milestone_every": 5, "keep_milestones": 2,
    "lr": 1e-4, "lr_encoder": 2e-5, "weight_decay": 1e-4, "max_grad_norm": 1.0,
    "warmup_epochs": 1, "gradient_checkpointing": True,
    "patience": 5, "min_delta": 1e-3,
    "log_every": 10,
    "out_dir": str(WORKING / "runs"), "local_dir": str(WORKING),
    "qacp_temperature": 0.1, "seed": 42,
}

qacp_path = WORKING / "qacp_config.yaml"
with open(qacp_path, "w") as f:
    yaml.dump(QACP_CONFIG, f)
print(f"QACP config -> {qacp_path}")


In [ ]:
# Cell 9: Run QACP Stage 0
!cd {REPO} && python -m src.training.pretrain_qacp \
    --config {qacp_path} \
    --run-id {QACP_CONFIG['run_id']}

In [ ]:
# Cell 10: Stage 1 config (init from QACP best checkpoint on HF) -- 3 seeds
import glob
from pathlib import Path

# -- Download the QACP best checkpoint from HF --------------------------------
from src.utils.hf_backup import HFBackup

QACP_RUN_ID = QACP_CONFIG["run_id"]   # "qacp_stage0"
_qacp_dl_dir = WORKING / "qacp_ckpt"
_qacp_dl_dir.mkdir(parents=True, exist_ok=True)

qacp_backup = HFBackup(run_id=QACP_RUN_ID, local_dir=str(WORKING))
print(f"Downloading QACP best checkpoint from HF (run_id={QACP_RUN_ID})...")

qacp_ckpt = qacp_backup.download_best(local_dir=str(_qacp_dl_dir))
if qacp_ckpt is None:
    print("  best.pt not found -- trying latest checkpoint...")
    qacp_ckpt = qacp_backup.download_latest(local_dir=str(_qacp_dl_dir))
if qacp_ckpt is None:
    print("WARNING: No QACP checkpoint found on HF.\n"
          "Stage 1 will train from RANDOM INIT (sub-optimal but safe).\n"
          "Re-run Cell 9 first to generate a QACP checkpoint.")
else:
    print(f"QACP checkpoint: {qacp_ckpt}")

def run_complete(run_id, epochs):
    """True when HF holds a finished run (all epochs, or QACP early-stop). Cheap: reads
    only resume_state.json. Every downstream cell gates on this so a partially trained
    run is never evaluated/published by a later 'Run All' session."""
    hb = HFBackup(run_id=run_id, local_dir=str(WORKING))
    ep = hb.peek_resume_epoch()
    if ep is None:
        return False
    if ep + 1 >= epochs:
        return True
    try:  # QACP early stopping writes early_stop into the log/checkpoint metrics
        import json as _j
        from huggingface_hub import hf_hub_download
        st = _j.load(open(hf_hub_download(hb.repo_id, f"{hb.base_path}/state/resume_state.json", repo_type="model")))
        return bool(st.get("no_improve", 0) >= 5 and "best_loss" in st)
    except Exception:
        return False

def best_ckpt_for(run_id):
    """Local runs/<run_id>/best.pt if this session trained it, else download from HF."""
    local = WORKING / "runs" / run_id / "best.pt"
    if local.exists():
        return local
    p = HFBackup(run_id=run_id, local_dir=str(WORKING)).download_best(local_dir=str(WORKING / "dl" / run_id))
    return Path(p) if p else None

# -- Build Stage-1 configs (3 seeds) ------------------------------------------
# Stage-1 TEMPLATE. run_id, seed and epochs come from the job this worker claims, so
# the same notebook runs unmodified on all 10 accounts -- edit the queue, not the
# notebook:  python scripts/publish_campaign.py jobs --seeds 42,123,456 --epochs 10
# ~15k train clips at batch 4 x accum 4 = 940 optimizer steps/epoch, ~50 min/epoch on
# a T4, so one 10-epoch seed is ~10 GPU-h and fits inside a single 12 h batch session.
STAGE1_BASE = {
"d_model": 768, "n_heads": 8, "n_fusion_layers": 4, "dropout": 0.1,
"use_sync": True, "use_disentangle": True, "compose_quadrant": False,
"video_backbone": "videomae", "audio_backbone": "wavlm",
"video_model_name": "MCG-NJU/videomae-base",
"audio_model_name": "microsoft/wavlm-base-plus",
"freeze_blocks": 6, "freeze_feature_extractor": True,
# Use downloaded QACP weights; None falls back to random init gracefully
"init_from": qacp_ckpt if (qacp_ckpt and Path(qacp_ckpt).exists()) else None,
"n_frames": 16, "audio_len": 64000, "shard_root": None,
"cache_root": CACHE_ROOT,        # packed clips; None = live ffmpeg decode
"feature_cache": None,
"train_manifest": str(TRAIN_MANIFEST),       # train split ONLY (val/test are held out)
"val_manifest": str(VAL_MANIFEST),
"root_dir": FAKEAV_ROOT,
"modality_dropout": 0.15, "augment": True,
# batch 8 x accum 2 = 16, the same effective batch as 4 x 4 and the same ~940
        # optimizer steps/epoch, but it splits 4-per-GPU on Kaggle's T4 x2 instead of
        # leaving a device idle. On a single T4 it is ~8.9 GB of 15.6, still fine.
        "batch_size": 8, "grad_accum_steps": 2, "num_workers": NUM_WORKERS,
        "epochs": 10,
        "data_parallel": True,   # DataParallel when >1 GPU is visible; see src/utils/parallel.py
"milestone_every": 5, "keep_milestones": 2,
"lr": 1e-4, "lr_encoder": 1e-5, "weight_decay": 1e-4, "max_grad_norm": 1.0,
"warmup_epochs": 1, "gradient_checkpointing": True,
"nan_patience": 5, "max_nan_restores": 3,
"log_every": 25,
"out_dir": str(WORKING / "runs"), "local_dir": str(WORKING),
"loss_weights": {"v": 1.0, "a": 1.0, "quad": 0.5, "loc": 0.5,
                 "sync": 0.1, "disentangle": 0.1},
}
print("Stage-1 template ready (init:",
      "from QACP)" if STAGE1_BASE["init_from"] else "random init)")

# One place that turns (seed, run_id, epochs) into a config file on disk. Cell 11 uses it
# for the job it claims; cells 12-19 use the list below.
def stage1_config_for(seed, run_id, epochs):
    cfg = dict(STAGE1_BASE)
    cfg.update(run_id=run_id, seed=seed, epochs=epochs)
    path = WORKING / f"{run_id}_config.yaml"
    with open(path, "w") as f:
        yaml.dump(cfg, f)
    return path, cfg

# Cells 12-19 consume STAGE1_CONFIGS as [(seed, config_path, cfg), ...]. Derive it from the
# published queue so evaluation/ablations automatically cover whatever seeds the campaign
# actually ran, instead of a hard-coded list that can drift from the queue.
from src.utils.coordinator import Coordinator as _Coordinator
_stage1_jobs = [j for j in _Coordinator().load_jobs() if j.kind == "stage1"]
if _stage1_jobs:
    STAGE1_CONFIGS = []
    for _j in sorted(_stage1_jobs, key=lambda j: j.priority):
        _seed = _j.config.get("seed", 42)
        _path, _cfg = stage1_config_for(_seed, _j.run_id, _j.epochs)
        STAGE1_CONFIGS.append((_seed, _path, _cfg))
else:
    print("no queue published yet -> single seed-42 config so downstream cells still work")
    _path, _cfg = stage1_config_for(42, "stage1_v3_seed42", 10)
    STAGE1_CONFIGS = [(42, _path, _cfg)]

print("Stage-1 campaign:", [(s, c["run_id"], c["epochs"]) for s, _, c in STAGE1_CONFIGS])

In [ ]:
# Cell 11: Stage 1 WORKER LOOP - run this identical notebook on every Kaggle account
#
# Each worker claims a job from coord/jobs.json on the shared HF repo, trains it, and
# pushes to runs/<run_id>/. A claim is a LEASE: if this session is reaped, another account
# reclaims the job after 45 min and resumes from the last checkpoint. BalancedBatchSampler
# is a pure function of (seed, epoch) with mid-epoch skip, so a job that migrates between
# accounts sees exactly the batches an uninterrupted run would have.
#
# Publish the queue once, from anywhere:
#   python scripts/publish_campaign.py jobs --seeds 42,123,456 --epochs 10
#   python scripts/publish_campaign.py status      # who is running what, all accounts
import subprocess, signal, sys, threading, time as _time, yaml
from src.utils.coordinator import Coordinator
from src.utils.hf_storage import maybe_reclaim

!free -g

COORD = Coordinator()
print(f"[worker] identity: {COORD.worker_id}")
for row in COORD.status():
    print(f"   [{row['state']:<7}] {row['job_id']:<22} {row['worker']}")

_NOISE = ("Loading weights", "Processing Files", "New Data Upload", "Downloading",
          "model.safetensors", "pytorch_model.bin", "config.json", "preprocessor_config.json")

def _keep(line):
    if "\r" in line:                      # tqdm / hf_hub progress-bar redraws
        return False
    return not line.lstrip().startswith(_NOISE)

while True:
    job = COORD.acquire(kinds=["stage1"])
    if job is None:
        print("[worker] queue drained - nothing left to claim. Done.")
        break

    claimed_at = _time.time()
    stop_beat = threading.Event()
    def _beat():
        while not stop_beat.wait(300):    # 5 min << the 45 min lease
            COORD.heartbeat(job.job_id, claimed_at=claimed_at)
    threading.Thread(target=_beat, daemon=True).start()

    print(f"\n{'='*60}\n[worker] training {job.job_id} (seed={job.config.get('seed')}, "
          f"{job.epochs} epochs) -> runs/{job.run_id}\n{'='*60}", flush=True)
    config_path, _ = stage1_config_for(job.config.get("seed", 42), job.run_id, job.epochs)
    env = dict(os.environ, PYTHONUNBUFFERED="1", DAVIDNET_SESSION_ID=SESSION_ID)
    proc = subprocess.Popen(["python", "-u", "-m", "src.training.train", "--config", str(config_path),
                             "--run-id", job.run_id], cwd=REPO, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    dropped, last_flush = 0, 0.0
    for line in proc.stdout:
        if not _keep(line):
            dropped += 1
            continue
        print(line, end="")
        now = _time.monotonic()
        if now - last_flush >= 1.0:       # at most ~1 websocket flush per second
            sys.stdout.flush()
            last_flush = now
    sys.stdout.flush()
    rc = proc.wait()
    stop_beat.set()
    print(f"   ({dropped} progress-bar lines suppressed)")

    hb = HFBackup(run_id=job.run_id, local_dir=str(WORKING))
    finished = hb.is_complete(job.epochs)
    if rc == 0 and finished:
        COORD.complete(job.job_id, provenance={
            "git_commit": subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO,
                                         capture_output=True, text=True).stdout.strip(),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            "run_type": os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "?"),
            "torch": torch.__version__, "worker": COORD.worker_id})
        print(f"[worker] {job.job_id} COMPLETE -> receipt written")
    else:
        # Partial progress is already on HF. Drop the lease so another account resumes
        # immediately instead of waiting out the 45 min expiry.
        why = f"signal {signal.Signals(-rc).name}" if rc < 0 else f"exit {rc}" if rc else "epochs remaining"
        COORD.withdraw(job.job_id, reason=why)
        print(f"[worker] {job.job_id} INCOMPLETE ({why}) - lease released for another account")

    maybe_reclaim(COORD)                  # fleet-wide: squash HF history at most every 6 h

In [ ]:
# Cell 12: In-domain test + cross-dataset evaluation (with HF backup)
import json

# In-domain FakeAVCeleb test split FIRST (Table 3.1), then cross-dataset (Table 3.2).
EVAL_DATASETS = {
    "fakeavceleb-test": (TEST_MANIFEST, FAKEAV_ROOT, 0),
    "celeb-df-v2":      (MANIFEST_DIR / "celeb-df-v2.jsonl", datasets.get("celeb-df-v2"), 0),
    "dfdc-10":          (MANIFEST_DIR / "dfdc-10.jsonl", datasets.get("dfdc-10"), 4000),
    "deepfaketimit":    (MANIFEST_DIR / "deepfaketimit.jsonl", datasets.get("deepfaketimit"), 0),
    "in-the-wild":      (MANIFEST_DIR / "in-the-wild.jsonl", datasets.get("in-the-wild"), 6000),
    "asvpoof-2019":     (MANIFEST_DIR / "asvpoof-2019.jsonl", datasets.get("asvpoof-2019"), 6000),
    "wavefake":         (MANIFEST_DIR / "wavefake.jsonl", datasets.get("wavefake"), 4000),
}
# max_clips > 0 = deterministic stratified subsample (WaveFake alone is 134k files)

all_results = {}
for seed, config_path, stage1_cfg in STAGE1_CONFIGS:
    run_id = stage1_cfg["run_id"]
    if not run_complete(run_id, stage1_cfg["epochs"]):
        print(f"Seed {seed}: training not finished yet on HF - evaluation deferred to a later session")
        continue
    best_ckpt = best_ckpt_for(run_id)
    if not best_ckpt:
        print(f"  No checkpoint on HF - skipping seed {seed}")
        continue

    seed_results = {}
    for ds_name, (manifest, root, max_clips) in EVAL_DATASETS.items():
        if manifest is None or not Path(manifest).exists() or root is None:
            print(f"  {ds_name}: no manifest/root, skipping")
            continue
        print(f"\nSeed {seed} on {ds_name}...")
        report_path = WORKING / f"eval_{run_id}_{ds_name}.json"
        !cd {REPO} && python -m src.eval.evaluate \
            --config {config_path} \
            --checkpoint {best_ckpt} \
            --manifest {manifest} \
            --root-dir {root} \
            --max-clips {max_clips} \
            --out {report_path} \
            --run-id {run_id} \
            --ds-name {ds_name} \
            --skip-if-done
        if report_path.exists():
            with open(report_path) as f:
                r = json.load(f)
            seed_results[ds_name] = {
                "video_auc": r["video"]["auc"],
                "audio_auc": r["audio"]["auc"],
                "quadrant_acc": r["quadrant"]["acc"],
                "n": r["n"],
            }
            print(f"  v_auc={r['video']['auc']} a_auc={r['audio']['auc']} quad_acc={r['quadrant']['acc']:.4f} n={r['n']}")
    all_results[f"seed{seed}"] = seed_results

print("\n" + "="*70)
print("EVALUATION SUMMARY  (AUC is NaN when a manifest has a single class for that modality)")
print("="*70)
print(f"{'Dataset':<20} {'Type':<8} {'V-AUC':<10} {'A-AUC':<10} {'Quad-Acc':<10} {'n':<8}")
print("-"*70)
for seed_key, ds_results in all_results.items():
    for ds, m in ds_results.items():
        dtype = "AV" if ds in ["fakeavceleb-test", "dfdc-10", "deepfaketimit", "celeb-df-v2"] else "Audio"
        va = "nan" if m['video_auc'] != m['video_auc'] else f"{m['video_auc']:.4f}"
        aa = "nan" if m['audio_auc'] != m['audio_auc'] else f"{m['audio_auc']:.4f}"
        print(f"{ds:<20} {dtype:<8} {va:<10} {aa:<10} {m['quadrant_acc']:<10.4f} {m['n']:<8}")


In [ ]:
# Cell 13: Experiment runner helpers (used by the baseline / ablation / LOGO cells below)
# Every extra experiment is ONE resumable run_id on HF: re-running a cell after a session
# death continues where it stopped; finished evals are skipped (--skip-if-done).
import yaml, json, copy
from pathlib import Path

BASE_CFG = STAGE1_CONFIGS[0][2]                     # seed-42 Stage-1 config (dict)
BASE_CFG_PATH = STAGE1_CONFIGS[0][1]

def eval_datasets_for(kind):
    """(ds_name, manifest, root, max_clips) list: in-domain test + the two cross-dataset
    corpora that can score each stream (Celeb-DF -> video, In-the-Wild -> audio)."""
    ds = [("fakeavceleb-test", TEST_MANIFEST, FAKEAV_ROOT, 0)]
    if kind == "ablation":
        ds += [("celeb-df-v2", MANIFEST_DIR / "celeb-df-v2.jsonl", datasets.get("celeb-df-v2"), 1500),
               ("in-the-wild", MANIFEST_DIR / "in-the-wild.jsonl", datasets.get("in-the-wild"), 3000)]
    return [d for d in ds if d[1] is not None and Path(d[1]).exists() and d[2] is not None]

def train_and_eval(run_id, overrides, evals):
    """Train (resumable) then evaluate on `evals`; eval JSONs land in WORKING as
    eval_<run_id>_<ds>.json and on HF under runs/<run_id>/eval/."""
    cfg = copy.deepcopy(BASE_CFG); cfg.update(overrides); cfg["run_id"] = run_id
    for k, v in overrides.items():
        if k.startswith("loss_weights."):
            cfg["loss_weights"][k.split(".", 1)[1]] = v
    cfg = {k: v for k, v in cfg.items() if not k.startswith("loss_weights.")}
    cfg_path = WORKING / f"{run_id}_config.yaml"
    with open(cfg_path, "w") as f:
        yaml.dump(cfg, f)
    print(f"\n{'='*60}\n{run_id}: epochs={cfg['epochs']} train={Path(cfg['train_manifest']).name} "
          f"init={'QACP' if cfg.get('init_from') else 'none'}\n{'='*60}")
    !cd {REPO} && python -m src.training.train --config {cfg_path} --run-id {run_id}
    if not run_complete(run_id, cfg["epochs"]):
        print(f"  {run_id}: not finished (session budget) - it will resume on the next Run All")
        return None
    ckpt = best_ckpt_for(run_id)
    if not ckpt:
        print(f"  {run_id}: finished but no best.pt on HF?! skipping eval")
        return None
    for ds_name, manifest, root, max_clips in evals:
        report_path = WORKING / f"eval_{run_id}_{ds_name}.json"
        !cd {REPO} && python -m src.eval.evaluate --config {cfg_path} --checkpoint {ckpt} \
            --manifest {manifest} --root-dir {root} --max-clips {max_clips} \
            --out {report_path} --run-id {run_id} --ds-name {ds_name} --skip-if-done
    return cfg_path


In [ ]:
# Cell 14: Explainability figures (Fig. C): saliency on frames + spectrogram, sync curve, localization
best_seed_cfg = None
for seed, config_path, cfg in STAGE1_CONFIGS:
    if run_complete(cfg["run_id"], cfg["epochs"]):
        ck = best_ckpt_for(cfg["run_id"])
        if ck:
            best_seed_cfg = (seed, config_path, cfg, ck)
            break
assert best_seed_cfg, "no trained checkpoint found - run Cell 11 first"
seed, config_path, cfg, best_ckpt = best_seed_cfg
assert run_complete(cfg["run_id"], cfg["epochs"]), "main run not finished - explain deferred"
EXPLAIN_DIR = WORKING / "explain"
!cd {REPO} && python -m src.eval.explain --config {config_path} --checkpoint {best_ckpt} \
    --manifest {TEST_MANIFEST} --root-dir {FAKEAV_ROOT} --out {EXPLAIN_DIR} --n 2


In [ ]:
# Cell 15: Baselines under the identical split/metrics (Table 1 rows + ROC panel)
# Budget: decode-bound (~5 clips/s) -> BASELINE_STEPS micro-batches of 8 per epoch.
RUN_BASELINES = True
BASELINES = ["audio-wavlm", "video-framecnn", "audio-speccnn"]   # + "video-xception" (needs `pip install timm`)
BASELINE_EPOCHS, BASELINE_STEPS = 3, 800                          # 3 x 800 x 8 = 19k clips seen per baseline

if RUN_BASELINES:
    _hb = HFBackup(run_id="baselines", local_dir=str(WORKING))
    for name in BASELINES:
        out = WORKING / f"baseline_{name}.json"
        if not out.exists() and _hb.has_file(f"paper/metrics/baseline_{name}.json"):
            got = _hb.download_file(f"paper/metrics/baseline_{name}.json", local_dir=str(WORKING / "dl"))
            if got:
                import shutil as _sh; _sh.copy2(got, out)
        if out.exists():
            print(f"{name}: done (from HF)" if not out.stat().st_size == 0 else f"{name}: done"); continue
        !cd {REPO} && python -m src.baselines.train_baseline --baseline {name} \
            --train-manifest {TRAIN_MANIFEST} --test-manifest {TEST_MANIFEST} --root-dir {FAKEAV_ROOT} \
            --epochs {BASELINE_EPOCHS} --max-steps-per-epoch {BASELINE_STEPS} --max-test-clips 1500 \
            --batch-size 8 --num-workers 4 --out {out}
        if out.exists():
            api_hf = HFBackup(run_id="baselines", local_dir=str(WORKING))._get_api()
            api_hf.upload_file(path_or_fileobj=str(out), path_in_repo=f"paper/metrics/baseline_{name}.json",
                               repo_id="MoshinAli/david-net-av-backup", repo_type="model")


In [ ]:
# Cell 16: Ablations (Table 5 / Fig. ablation) - seed 42, reduced epochs, one run_id each
# Each variant costs ~ABL_EPOCHS x (epoch time of the main run). Order = importance.
RUN_ABLATIONS = True
ABL_EPOCHS = 4
ABLATIONS = {
    "no_qacp":        {"init_from": None},
    "no_sync":        {"use_sync": False, "loss_weights.sync": 0.0},
    "no_disentangle": {"use_disentangle": False, "loss_weights.disentangle": 0.0},
    "compose_quad":   {"compose_quadrant": True},
    # "no_loc":       {"loss_weights.loc": 0.0},
    # "no_moddrop":   {"modality_dropout": 0.0},
    # "single_task":  {"loss_weights.quad": 0.0, "loss_weights.loc": 0.0, "loss_weights.sync": 0.0, "loss_weights.disentangle": 0.0},
}
ABLATION_RUN_IDS = [f"abl_{n}_v2_seed42" for n in ABLATIONS]
if RUN_ABLATIONS:
    for name, ov in ABLATIONS.items():
        train_and_eval(f"abl_{name}_v2_seed42", dict(ov, epochs=ABL_EPOCHS, seed=42), eval_datasets_for("ablation"))


In [ ]:
# Cell 17: Leave-one-generator-FAMILY-out (Table 3) - generator- AND subject-disjoint
# Families: wav2lip (incl. faceswap-/fsgan-wav2lip hybrids), fsgan, faceswap, rtvc.
RUN_LOGO = True
LOGO_EPOCHS = 4
LOGO_FAMILIES = ["wav2lip", "fsgan", "faceswap", "rtvc"]
LOGO_RUN_IDS = [f"logo_{f}_v2_seed42" for f in LOGO_FAMILIES]
if RUN_LOGO:
    for fam in LOGO_FAMILIES:
        tr, va, te = (FAKEAV_SPLITS / f"logo_{fam}_{s}.jsonl" for s in ("train", "val", "test"))
        if not tr.exists():
            print(f"{fam}: no LOGO split (re-run Cell 6)"); continue
        train_and_eval(f"logo_{fam}_v2_seed42",
                       {"train_manifest": str(tr), "val_manifest": str(va), "epochs": LOGO_EPOCHS, "seed": 42},
                       [(f"logo-{fam}", te, FAKEAV_ROOT, 0)])


In [ ]:
# Cell 18: DAVID-Net-Lite (deployable variant, Table 7) — QACP-lite, then Stage-1 with logit
# distillation from the full model's seed-42 best.pt. ~3x cheaper per epoch than the full model.
RUN_LITE = True
LITE_QACP_EPOCHS, LITE_EPOCHS = 8, 10
LITE_RUN_ID = "stage1_lite_v2_seed42"
lite_base = yaml.safe_load(open(Path(REPO) / "configs" / "david_net_lite.yaml"))
lite_base.update({"root_dir": FAKEAV_ROOT, "train_manifest": str(TRAIN_MANIFEST), "val_manifest": str(VAL_MANIFEST),
                  "out_dir": str(WORKING / "runs"), "local_dir": str(WORKING),
                  "num_workers": NUM_WORKERS, "cache_root": CACHE_ROOT})
LITE_CFG_PATH = WORKING / f"{LITE_RUN_ID}_config.yaml"
MAIN42 = STAGE1_CONFIGS[0][2]
if RUN_LITE and not run_complete(MAIN42["run_id"], MAIN42["epochs"]):
    print("Lite deferred: the full seed-42 model (distillation teacher) is not finished yet")
    RUN_LITE = False
if RUN_LITE:
    # --- Stage 0 (lite)
    qacp_lite = dict(lite_base, run_id="qacp_lite_v2", epochs=LITE_QACP_EPOCHS, batch_size=8, grad_accum_steps=4,
                     freeze_blocks=8, freeze_blocks_audio=0, qacp_views_per_clip=4, qacp_temperature=0.2,
                     patience=5, min_delta=1e-3, lr=1e-4, lr_encoder=1e-5, warmup_epochs=1, log_every=10)
    qacp_lite_path = WORKING / "qacp_lite_config.yaml"
    yaml.dump(qacp_lite, open(qacp_lite_path, "w"))
    !cd {REPO} && python -m src.training.pretrain_qacp --config {qacp_lite_path} --run-id qacp_lite_v2
    lite_qacp_ckpt = HFBackup(run_id="qacp_lite_v2", local_dir=str(WORKING)).download_best(local_dir=str(WORKING / "qacp_lite_ckpt"))
    # --- teacher = full model, seed 42
    teacher = WORKING / "runs" / STAGE1_CONFIGS[0][2]["run_id"] / "best.pt"
    if not teacher.exists():
        teacher = HFBackup(run_id=STAGE1_CONFIGS[0][2]["run_id"], local_dir=str(WORKING)).download_best(
            local_dir=str(WORKING / "dl" / STAGE1_CONFIGS[0][2]["run_id"]))
    lite_cfg = dict(lite_base, run_id=LITE_RUN_ID, epochs=LITE_EPOCHS, seed=42,
                    init_from=lite_qacp_ckpt, distill_from=str(teacher) if teacher and Path(str(teacher)).exists() else None)
    if not lite_cfg["distill_from"]:
        print("WARNING: no full-model checkpoint found -> Lite trains WITHOUT distillation")
    yaml.dump(lite_cfg, open(LITE_CFG_PATH, "w"))
    # --- Stage 1 (lite) + evaluation on every corpus of Cell 12
    !cd {REPO} && python -m src.training.train --config {LITE_CFG_PATH} --run-id {LITE_RUN_ID}
    if not run_complete(LITE_RUN_ID, LITE_EPOCHS):
        raise SystemExit("Lite not finished in this session - it resumes on the next Run All")
    LITE_CKPT = best_ckpt_for(LITE_RUN_ID)
    for ds_name, (manifest, root, max_clips) in EVAL_DATASETS.items():
        if manifest is None or not Path(manifest).exists() or root is None:
            continue
        report_path = WORKING / f"eval_{LITE_RUN_ID}_{ds_name}.json"
        !cd {REPO} && python -m src.eval.evaluate --config {LITE_CFG_PATH} --checkpoint {LITE_CKPT} \
            --manifest {manifest} --root-dir {root} --max-clips {max_clips} \
            --out {report_path} --run-id {LITE_RUN_ID} --ds-name {ds_name} --skip-if-done


In [ ]:
# Cell 19: Robustness sweep (Fig. A, RQ5) + ALL manuscript artifacts -> HF paper/
# Gated on a COMPLETED Stage-1 run; robustness JSON is reused from HF when present.
best_seed_cfg = None
for seed, config_path, cfg in STAGE1_CONFIGS:
    if run_complete(cfg["run_id"], cfg["epochs"]):
        ck = best_ckpt_for(cfg["run_id"])
        if ck:
            best_seed_cfg = (seed, config_path, cfg, ck)
            break
assert best_seed_cfg, "no COMPLETED Stage-1 run on HF yet - artifacts deferred to a later session"
seed, config_path, cfg, best_ckpt = best_seed_cfg
ROBUSTNESS_JSON = WORKING / f"robustness_{cfg['run_id']}.json"
_hb = HFBackup(run_id=cfg["run_id"], local_dir=str(WORKING))
if not ROBUSTNESS_JSON.exists() and _hb.has_file("paper/metrics/robustness.json"):
    got = _hb.download_file("paper/metrics/robustness.json", local_dir=str(WORKING / "dl"))
    if got:
        import shutil as _sh; _sh.copy2(got, ROBUSTNESS_JSON)
if not ROBUSTNESS_JSON.exists():
    !cd {REPO} && python -m src.eval.robustness \
        --config {config_path} --checkpoint {best_ckpt} \
        --manifest {TEST_MANIFEST} --root-dir {FAKEAV_ROOT} --max-clips 600 \
        --out {ROBUSTNESS_JSON}

# only FINISHED extra runs feed the tables; unfinished ones simply resume next session
RUN_IDS = " ".join(c["run_id"] for _, _, c in STAGE1_CONFIGS if run_complete(c["run_id"], c["epochs"]))
ABL_DONE = " ".join(r for r in globals().get("ABLATION_RUN_IDS", []) if run_complete(r, globals().get("ABL_EPOCHS", 4)))
LOGO_DONE = " ".join(r for r in globals().get("LOGO_RUN_IDS", []) if run_complete(r, globals().get("LOGO_EPOCHS", 4)))
LITE_ARGS = ""
if globals().get("LITE_RUN_ID") and run_complete(LITE_RUN_ID, globals().get("LITE_EPOCHS", 10)):
    LITE_ARGS = f"--lite-run-id {LITE_RUN_ID} --lite-config {LITE_CFG_PATH} --lite-checkpoint {best_ckpt_for(LITE_RUN_ID)}"
EXPLAIN_ARGS = f"--explain-dir {WORKING}/explain" if (WORKING / "explain").exists() else ""
ABL_ARGS = f"--ablation-run-ids {ABL_DONE}" if ABL_DONE else ""
LOGO_ARGS = f"--logo-run-ids {LOGO_DONE}" if LOGO_DONE else ""
print(f"aggregating: seeds=[{RUN_IDS}] ablations=[{ABL_DONE}] logo=[{LOGO_DONE}] lite={'yes' if LITE_ARGS else 'no'}")

!cd {REPO} && python scripts/paper_artifacts.py \
    --work {WORKING} --out {WORKING}/paper \
    --run-ids {RUN_IDS} --qacp-run-id {QACP_CONFIG['run_id']} \
    --in-domain fakeavceleb-test \
    --robustness {ROBUSTNESS_JSON} --config {config_path} --checkpoint {best_ckpt} \
    {EXPLAIN_ARGS} {ABL_ARGS} {LOGO_ARGS} {LITE_ARGS} \
    --push
# -> HF: paper/metrics/{summary, per_generator, efficiency, fairness, baselines, ablation, logo}.json, tables.tex, eval/,
#        paper/figures/results_{roc, roc_cross_dataset, reliability, confusion, localization,
#        robustness, training_curves, per_generator, ablation, explain_*}.{pdf,png}, paper/logs/*, paper/configs/*


In [ ]:
# Cell 20: Publish the model for END USERS -> public HF model repo (weights + config + model card + API)
# The backup repo is private and full of optimizer states; users get MoshinAli/david-net-av.
PUBLIC_MODEL_REPO = "MoshinAli/david-net-av"
assert run_complete(cfg["run_id"], cfg["epochs"]), "full model not finished - publish deferred"
!cd {REPO} && python scripts/publish_model.py     --checkpoint {best_ckpt} --config {config_path}     --summary {WORKING}/paper/metrics/summary.json     --repo {PUBLIC_MODEL_REPO} --public --repo-root {REPO}
print(f"Model card: https://huggingface.co/{PUBLIC_MODEL_REPO}")

# DAVID-Net-Lite -> the repo the FastAPI Space pulls by default (api/Dockerfile)
if globals().get("LITE_RUN_ID") and run_complete(LITE_RUN_ID, globals().get("LITE_EPOCHS", 10)):
    LITE_BEST = best_ckpt_for(LITE_RUN_ID)
    !cd {REPO} && python scripts/publish_model.py \
        --checkpoint {LITE_BEST} --config {LITE_CFG_PATH} \
        --summary {WORKING}/paper/metrics/summary.json \
        --repo MoshinAli/david-net-av-lite --public --repo-root {REPO}
    print("Lite model card: https://huggingface.co/MoshinAli/david-net-av-lite")


In [ ]:
# Cell 21: Final summary (from paper/metrics/summary.json)
import json
S = json.load(open(WORKING / "paper" / "metrics" / "summary.json"))
print("="*78)
print("DAVID-Net — results (mean +/- std over seeds", S["run_ids"], ")")
print("="*78)
print(f"{'Dataset':<22}{'n':>7}{'V-AUC':>16}{'A-AUC':>16}{'Quad-Acc':>16}")
for ds, d in S["datasets"].items():
    a = d["aggregate"]; n = next(iter(d["per_seed"].values()))["n"]
    f = lambda k: "--" if a[k]["mean"] != a[k]["mean"] else f"{a[k]['mean']:.4f}+/-{a[k]['std']:.4f}"
    tag = "*" if ds == S["in_domain"] else " "
    print(f"{ds+tag:<22}{n:>7}" + f("video_auc").rjust(16) + f("audio_auc").rjust(16) + f("quadrant_acc").rjust(16))
print("* = in-domain test split; -- = single-class corpus for that stream")
print()
print("On HF (MoshinAli/david-net-av-backup):")
print("  runs/<run_id>/{checkpoints,best,logs,eval,state}   per-run training state")
print("  paper/metrics/   summary.json, per_generator.json, efficiency.json, tables.tex, eval/")
print("  paper/figures/   ROC (in-domain + cross-dataset), reliability, confusion, localization,")
print("                   robustness, training curves, per-generator")
print("  paper/logs/, paper/configs/")
print("Public model for end users: https://huggingface.co/MoshinAli/david-net-av")
